# Dataset extraction on a Colab A100

Runs the dataset pipeline **inside Colab**, next to the model server: no tunnel, no image uploads over
the internet, and 32 requests in flight so vLLM can batch them.

1. Add a secret named `VLM_API_KEY` (key icon in the left sidebar) with a long random value.
2. Put your PDFs in a Google Drive folder, e.g. `MyDrive/green_pdfs/`.
3. Run the cells in order. If the session disconnects, run them again: finished papers are skipped.

In [ ]:
!pip install -q vllm
!apt-get -qq install -y default-jre-headless > /dev/null   # Java for OPSIN (name -> structure, offline)

In [ ]:
import os, subprocess, time, requests
from google.colab import userdata
API_KEY = userdata.get("VLM_API_KEY")          # never write the key in the notebook
os.environ["VLM_API_KEY"] = API_KEY
MODEL = "Qwen/Qwen2.5-VL-7B-Instruct"

server = subprocess.Popen(
    ["vllm", "serve", MODEL, "--dtype", "bfloat16", "--max-model-len", "16384",
     "--max-num-seqs", "64",                 # how many requests the GPU works on together
     "--gpu-memory-utilization", "0.90",
     "--api-key", API_KEY, "--port", "8000"],
    stdout=open("vllm.log", "w"), stderr=subprocess.STDOUT)

for _ in range(120):
    try:
        if requests.get("http://localhost:8000/v1/models",
                        headers={"Authorization": f"Bearer {API_KEY}"}, timeout=3).ok:
            print("Server ready"); break
    except Exception:
        pass
    time.sleep(5)
else:
    print(open("vllm.log").read()[-3000:])

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

# The repository is private: use a GitHub token with read access, or upload a zip of the repo instead.
REPO = "https://github.com/mohity24k/new-rg-calculator.git"
BRANCH = "claude/great-cerf-wh9q63"
!rm -rf /content/app && git clone -q -b {BRANCH} {REPO} /content/app
!pip install -q -r /content/app/requirements.txt pyyaml
%cd /content/app

## 1. Benchmark first (about 10-30 minutes)

Times the current app pipeline on 5 pages and the new pipeline at 1, 8 and 32 requests in flight.
Add `--gold benchmarks/gold.json` once you have hand-checked a few reactions (see `benchmarks/gold_template.json`).

In [ ]:
PDF_DIR = "/content/drive/MyDrive/green_pdfs"
import glob
sample = sorted(glob.glob(f"{PDF_DIR}/*.pdf"))[:2]
args = " ".join(f'--pdf "{p}"' for p in sample)
!python scripts/benchmark_extraction.py {args} --concurrency 1,8,32 --classic-pages 5 --report /content/drive/MyDrive/benchmark_report.json

## 2. Extract the whole folder (resumable)

Results are written to Drive after every paper. Re-run this cell after a disconnect.

In [ ]:
OUT = "/content/drive/MyDrive/green_dataset/run1"
!python scripts/build_dataset.py extract --pdf-dir "{PDF_DIR}" --out "{OUT}" --model {MODEL} --concurrency 32 --batch-papers 8

## 3. Structures, masses and checks (CPU; can also run later on any machine)

In [ ]:
!python scripts/build_dataset.py enrich --out "{OUT}"
!python scripts/build_dataset.py stats --out "{OUT}"

## Optional: serve the app's Upload PDF page through a tunnel

Only needed for the interactive app. The quick tunnel URL is public: keep the key secret.

In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared
import re
tunnel = subprocess.Popen(["./cloudflared", "tunnel", "--url", "http://localhost:8000"],
                          stdout=open("tunnel.log", "w"), stderr=subprocess.STDOUT)
for _ in range(30):
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open("tunnel.log").read())
    if m:
        print("Base URL for the app:", m.group(0) + "/v1"); break
    time.sleep(2)